In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv('updated3.csv')

print("Dataset Overview:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nTarget variable distribution:")
print(df['muscle_size_increase_cm2'].describe())



Dataset Overview:
   age gender exercise_type  sets  reps  weight  frequency  protein  calories  \
0   56      M     Deadlifts     6    15   146.1          4     90.5      2986   
1   46      M     Deadlifts     3    13   155.6          5    192.4      2794   
2   32      F        Squats     6     8   177.1          1    179.1      2245   
3   25      M        Squats     6     7   110.0          1    175.0      2057   
4   38      M     Deadlifts     5    11   151.4          5    104.1      3611   

   sleep    experience  muscle_size_increase_cm2 target_muscle_group  \
0    5.2      Advanced                      1.84                Back   
1    9.4  Intermediate                      4.61                Back   
2    5.1  Intermediate                      4.72               Quads   
3    9.7  Intermediate                      3.21               Quads   
4    7.4  Intermediate                      5.27                Back   

  exercise_category  
0          Compound  
1          Compoun

In [8]:
# Define baseline muscle growth equation parameters
def get_baseline_params(age, gender, experience):
    """Get baseline parameters for muscle growth equation"""
    
    # Genetic limit (M_max) - simplified based on gender and age
    if gender == 'M':
        base_limit = 45  # pounds of potential muscle gain
        age_factor = max(0.7, 1 - (age - 25) * 0.01)  # decline after 25
    else:  # Female
        base_limit = 30  # pounds of potential muscle gain  
        age_factor = max(0.7, 1 - (age - 25) * 0.008)
    
    M_max = base_limit * age_factor
    
    # Growth rate constant (k) based on experience
    k_values = {
        'Beginner': 0.20,
        'Intermediate': 0.10, 
        'Advanced': 0.05
    }
    k = k_values.get(experience, 0.10)
    
    return M_max, k

def calculate_baseline_growth(row, time_months=3):
    """Calculate baseline muscle growth using the equation"""
    M_max, k = get_baseline_params(row['age'], row['gender'], row['experience'])
    
    # Starting muscle size (M0) - assume some baseline
    M0 = 0
    
    # Baseline growth equation: M(t) = M0 + (M_max - M0)(1 - e^(-kt))
    # For monthly growth, we calculate growth rate
    baseline_growth = M_max * (1 - np.exp(-k * time_months))
    
    # Convert to cm2 approximation (very rough conversion)
    # This is a simplified conversion - in practice you'd need actual measurement relationships
    baseline_cm2 = baseline_growth * 0.2  # rough scaling factor
    
    return baseline_cm2

# Calculate baseline growth for each row
df['baseline_growth'] = df.apply(calculate_baseline_growth, axis=1)

# Calculate adjustment factor (what ML will predict)
df['adjustment_factor'] = df['muscle_size_increase_cm2'] / df['baseline_growth']

print("\nBaseline vs Actual Growth:")
print(f"Mean baseline growth: {df['baseline_growth'].mean():.2f} cm2")
print(f"Mean actual growth: {df['muscle_size_increase_cm2'].mean():.2f} cm2")
print(f"Mean adjustment factor: {df['adjustment_factor'].mean():.2f}")

# Feature engineering
le_gender = LabelEncoder()
le_exercise = LabelEncoder()
le_experience = LabelEncoder()
le_muscle_group = LabelEncoder()
le_category = LabelEncoder()

df['gender_encoded'] = le_gender.fit_transform(df['gender'])
df['exercise_type_encoded'] = le_exercise.fit_transform(df['exercise_type'])
df['experience_encoded'] = le_experience.fit_transform(df['experience'])
df['muscle_group_encoded'] = le_muscle_group.fit_transform(df['target_muscle_group'])
df['category_encoded'] = le_category.fit_transform(df['exercise_category'])

# Create additional features
df['protein_per_kg'] = df['protein'] / (df['weight'] * 0.45)  # rough bodyweight estimation
df['volume'] = df['sets'] * df['reps']
df['intensity'] = df['weight'] / df['reps']  # rough intensity measure
df['calories_per_kg'] = df['calories'] / (df['weight'] * 0.45)

# Select features for ML model
features = [
    'age', 'gender_encoded', 'exercise_type_encoded', 'sets', 'reps', 'weight',
    'frequency', 'protein', 'calories', 'sleep', 'experience_encoded',
    'muscle_group_encoded', 'category_encoded', 'protein_per_kg', 'volume',
    'intensity', 'calories_per_kg'
]

X = df[features]
y = df['adjustment_factor']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Performance:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))



Baseline vs Actual Growth:
Mean baseline growth: 1.93 cm2
Mean actual growth: 4.84 cm2
Mean adjustment factor: 2.59

Model Performance:
Mean Squared Error: 1.2187
R² Score: -3.7506

Top 10 Most Important Features:
                  feature  importance
14                 volume    0.186741
11   muscle_group_encoded    0.157182
8                calories    0.110943
5                  weight    0.077084
15              intensity    0.060543
9                   sleep    0.059581
2   exercise_type_encoded    0.055131
0                     age    0.051939
16        calories_per_kg    0.048642
6               frequency    0.045951


In [9]:

# Function to make predictions
def predict_muscle_growth(age, gender, exercise_type, sets, reps, weight,
                          frequency, protein, calories, sleep, experience,
                          target_muscle_group, exercise_category):
    """
    Predict muscle growth using baseline equation + ML adjustment
    """

    # Create input row
    input_data = pd.DataFrame({
        'age': [age],
        'gender': [gender],
        'exercise_type': [exercise_type],
        'sets': [sets],
        'reps': [reps],
        'weight': [weight],
        'frequency': [frequency],
        'protein': [protein],
        'calories': [calories],
        'sleep': [sleep],
        'experience': [experience],
        'target_muscle_group': [target_muscle_group],
        'exercise_category': [exercise_category]
    })

    # Calculate baseline growth
    baseline = calculate_baseline_growth(input_data.iloc[0])

    # Encode categorical variables
    input_data['gender_encoded'] = le_gender.transform(input_data['gender'])
    input_data['exercise_type_encoded'] = le_exercise.transform(
        input_data['exercise_type'])
    input_data['experience_encoded'] = le_experience.transform(
        input_data['experience'])
    input_data['muscle_group_encoded'] = le_muscle_group.transform(
        input_data['target_muscle_group'])
    input_data['category_encoded'] = le_category.transform(
        input_data['exercise_category'])

    # Create additional features
    input_data['protein_per_kg'] = input_data['protein'] / \
        (input_data['weight'] * 0.45)
    input_data['volume'] = input_data['sets'] * input_data['reps']
    input_data['intensity'] = input_data['weight'] / input_data['reps']
    input_data['calories_per_kg'] = input_data['calories'] / \
        (input_data['weight'] * 0.45)

    # Get features for prediction
    X_input = input_data[features]

    # Predict adjustment factor
    adjustment = rf_model.predict(X_input)[0]

    # Final prediction
    final_prediction = baseline * adjustment

    return {
        'baseline_growth': baseline,
        'adjustment_factor': adjustment,
        'final_prediction': final_prediction
    }

In [10]:

# Example prediction
example_result = predict_muscle_growth(
    age=19, gender='M', exercise_type='Squats', sets=4, reps=10, weight=80,
    frequency=3, protein=140, calories=2800, sleep=8, experience='Advanced',
    target_muscle_group='Quads', exercise_category='Compound'
)

print(f"\nExample Prediction:")
print(f"Baseline growth: {example_result['baseline_growth']:.2f} cm²")
print(f"Adjustment factor: {example_result['adjustment_factor']:.2f}")
print(f"Final prediction: {example_result['final_prediction']:.2f} cm²")


Example Prediction:
Baseline growth: 1.33 cm²
Adjustment factor: 1.76
Final prediction: 2.34 cm²


In [11]:
plt.figure(figsize=(8, 6))
plt.scatter(df['muscle_size_increase_cm2'], final_predictions, alpha=0.7)
plt.plot([df['muscle_size_increase_cm2'].min(), df['muscle_size_increase_cm2'].max()],
         [df['muscle_size_increase_cm2'].min(), df['muscle_size_increase_cm2'].max()], 'r--')
plt.xlabel('actual muscle size increase')
plt.ylabel('predicted muscle size increase')
plt.title('Actual vs Predicted Muscle Size Increase')
plt.show()

NameError: name 'final_predictions' is not defined

<Figure size 800x600 with 0 Axes>

In [ ]:
# Visualizations
plt.figure(figsize=(15, 10))

# Plot 1: Baseline vs Actual
plt.subplot(2, 3, 1)
plt.scatter(df['baseline_growth'], df['muscle_size_increase_cm2'], alpha=0.6)
plt.plot([df['baseline_growth'].min(), df['baseline_growth'].max()],
         [df['baseline_growth'].min(), df['baseline_growth'].max()], 'r--')
plt.xlabel('Baseline Growth (cm²)')
plt.ylabel('Actual Growth (cm²)')
plt.title('Baseline vs Actual Growth')

# Plot 2: Adjustment factors
plt.subplot(2, 3, 2)
plt.hist(df['adjustment_factor'], bins=20, alpha=0.7)
plt.xlabel('Adjustment Factor')
plt.ylabel('Frequency')
plt.title('Distribution of Adjustment Factors')

# Plot 3: Predicted vs Actual
plt.subplot(2, 3, 3)
y_pred_full = rf_model.predict(X)
plt.plot(y, y_pred_full, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('Actual Adjustment Factor')
plt.ylabel('Predicted Adjustment Factor')
plt.title('Model Predictions vs Actual')

# Plot 4: Feature importance
plt.subplot(2, 3, 4)
top_features = feature_importance.head(8)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top Feature Importances')

# Plot 5: Final predictions
plt.subplot(2, 3, 5)
final_predictions = df['baseline_growth'] * y_pred_full
plt.scatter(df['muscle_size_increase_cm2'], final_predictions, alpha=0.6)
plt.plot([df['muscle_size_increase_cm2'].min(), df['muscle_size_increase_cm2'].max()],
         [df['muscle_size_increase_cm2'].min(), df['muscle_size_increase_cm2'].max()], 'r--')
plt.xlabel('Actual Growth (cm²)')
plt.ylabel('Final Predicted Growth (cm²)')
plt.title('Final Predictions vs Actual')

# Plot 6: Residuals
plt.subplot(2, 3, 6)
residuals = df['muscle_size_increase_cm2'] - final_predictions
plt.scatter(final_predictions, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Growth (cm²)')
plt.ylabel('Residuals')
plt.title('Residuals Plot')

plt.tight_layout()
plt.show()

print("\nModel Training Complete!")
print("The model learns to adjust the baseline muscle growth equation based on individual factors.")
print("Use the predict_muscle_growth() function to make new predictions.")